# Build an AI Agent Assistant with LangChain Tool Calling

USE OF `AGENTS and TOOLS`

**ReAct framework for Agents**: 

A logical loop of:\
       - **Reason** → Think about the task.\
       - **Act** → Use a tool to perform an action.\
       - **Observe** → Check the tool's output.\
       - **Plan** → Decide what to do next.

In [ ]:
from langchain_openai import ChatOpenAI

from langchain.agents import Tool # Old way to create tool
from langchain_core.tools import tool # Suggested way to create tool
from langchain_community.utilities import WikipediaAPIWrapper

from langgraph.prebuilt import create_react_agent # For creating agents.
# from langchain.agents import create_agent # Its supported in langchain V1. and create_react_agent from langgraph is deprecated

from typing import List, Dict, Union
import re

from dotenv import load_dotenv
load_dotenv()

True

In [7]:
openai_llm = ChatOpenAI(model="gpt-4.1-nano")


## Tool Creation - Old way

In [ ]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input by extracting numbers from a string and return result.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (String):
    "10, 20 and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    result = sum(numbers)
    return {"result": result}

In [ ]:
print(add_numbers("1 2, 3"))
print(add_numbers("1 2 and 3"))
print(add_numbers("1 and 2 and 3"))
print(add_numbers("1 and 2and3"))

{'result': 6}
{'result': 6}
{'result': 6}
{'result': 1}


In [15]:
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object --> ",add_tool)

tool object -->  name='AddTool' description='Adds a list of numbers and returns the result.' func=<function add_numbers at 0x7fe999cff6a0>


In [16]:
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.func)

# Tool invoke
print("Tool Invoke:")
print(add_tool.func)

Tool Name:
AddTool
Tool Description:
Adds a list of numbers and returns the result.
Tool Function:
<function add_numbers at 0x7fe999cff6a0>
Tool Invoke:
<function add_numbers at 0x7fe999cff6a0>


### Calling Tool Function

In [17]:
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))

{'result': 60}


## Tool creation (@tool operator) - Recommended way

In [9]:
@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input by extracting numbers from a string and return result.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (String):
    "10, 20 and 30."

    Example Output:
    {"result": 60}
    """

    numbers = [int(num) for num in re.findall(r'-?\d+', inputs)] # Use regular expressions to extract all numbers from the input
    # numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    result = sum(numbers)
    return {"result": result}

In [33]:
print("Name: \n", add_numbers.name)
print("Args: \n", add_numbers.args) 
print("Description: \n", add_numbers.description) 


Name: 
 add_numbers
Args: 
 {'inputs': {'title': 'Inputs', 'type': 'string'}}
Description: 
 Adds a list of numbers provided in the input by extracting numbers from a string and return result.

Parameters:
- inputs (str): 
string, it should contain numbers that can be extracted and summed.

Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.

Example Input (String):
"10, 20 and 30."

Example Output:
{"result": 60}


In [88]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))

print(add_numbers.invoke("-10 -20 -30"))

{'result': 60}
{'result': -60}


### Multipe arguments in tool function

In [ ]:
@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

In [38]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

-6.2
6.2


### Multiple return types for tool functions

In [41]:
@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

## Create Agents

In [10]:
agent_exec = create_react_agent(model=openai_llm, tools=[add_numbers])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [69]:
msgs

{'messages': [HumanMessage(content='Add the numbers -10, -20, -30', id='a9b1c902-afaf-406e-875f-928afefbd74b'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HsLu6TAd69Mlj1PXpMR8Xwy4', 'function': {'arguments': '{"inputs":"-10, -20, -30"}', 'name': 'add_numbers'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 134, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_de604bd877', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-51cd37ca-596c-43f2-a55d-a0488b6e89c9-0', tool_calls=[{'name': 'add_numbers', 'args': {'inputs': '-10, -20, -30'}, 'id': 'call_HsLu6TAd69Mlj1PXpMR8Xwy4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 134, 'output_tokens': 21, 'total

In [70]:
print(msgs["messages"][-1].content)

The sum of -10, -20, and -30 is -60.


In [89]:
# Use the agent
response =agent_exec.invoke({"messages": [("human", "In 2023, the US GDP was approximately $27 trillion, while Canada's was around $2 trillion and Mexico's was about $1 trillion what is the total.")]})

In [90]:
response

{'messages': [HumanMessage(content="In 2023, the US GDP was approximately $27 trillion, while Canada's was around $2 trillion and Mexico's was about $1 trillion what is the total.", id='f698e6e2-7ab8-4c74-9f9f-9ebbd1587b8a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_TtlP3AvWIjuDygEm2LVVGByn', 'function': {'arguments': '{"inputs": "27 trillion, 2 trillion, 1 trillion"}', 'name': 'add_numbers'}, 'type': 'function'}, {'id': 'call_aAEGc9jE5Py2qPldCT3qsCl0', 'function': {'arguments': '{"inputs": "27, 2, 1"}', 'name': 'add_numbers'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 157, 'total_tokens': 216, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_de604bd877', 'finish_reason': 'tool_c

In [91]:
print(response["messages"][-1].content)

The total combined GDP of the US, Canada, and Mexico in 2023 was approximately $30 trillion.


## Using wikipedia tool

In [93]:
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [97]:
result = search_wikipedia.invoke("What is age of Imran Khan and Nawaz Sharif?")

In [104]:
result

"Page: Shehbaz Sharif\nSummary: Mian Muhammad Shehbaz Sharif (born 23 September 1951) is a Pakistani politician and businessman who has served as the 20th prime minister of Pakistan since March 2024, having previously been in the role between April 2022 to August 2023. He has also served as the president of the Pakistan Muslim League (N) and chief minister of Punjab three times, making him the longest-serving person in the role.\nSharif was elected to the Punjab Assembly in 1988 and to the National Assembly of Pakistan in 1990. He was re-elected to the Punjab Assembly in 1993 and appointed leader of the opposition. He was elected as chief minister of Pakistan's most populous province, Punjab, for the first time on 20 February 1997. After the 1999 Pakistani coup d'état, Sharif, along with his family, spent years of self-exile in Saudi Arabia, returning to Pakistan in 2007. Sharif was appointed chief minister for a second term after the PML(N)'s victory in Punjab in the 2008 Pakistani ge

In [103]:
result.split("\n\n\n\n")

["Page: Shehbaz Sharif\nSummary: Mian Muhammad Shehbaz Sharif (born 23 September 1951) is a Pakistani politician and businessman who has served as the 20th prime minister of Pakistan since March 2024, having previously been in the role between April 2022 to August 2023. He has also served as the president of the Pakistan Muslim League (N) and chief minister of Punjab three times, making him the longest-serving person in the role.\nSharif was elected to the Punjab Assembly in 1988 and to the National Assembly of Pakistan in 1990. He was re-elected to the Punjab Assembly in 1993 and appointed leader of the opposition. He was elected as chief minister of Pakistan's most populous province, Punjab, for the first time on 20 February 1997. After the 1999 Pakistani coup d'état, Sharif, along with his family, spent years of self-exile in Saudi Arabia, returning to Pakistan in 2007. Sharif was appointed chief minister for a second term after the PML(N)'s victory in Punjab in the 2008 Pakistani g

In [113]:
openai_llm2 = ChatOpenAI(model="gpt-4o-mini")

In [114]:
toolkit = [add_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
agent_updated = create_react_agent(
    model=openai_llm2,
    tools=toolkit,
    prompt="You are a helpful assistant that can perform add operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

In [115]:
query = "What is the population of India and Pakistan? Add populations of both"
response = agent_updated.invoke({"messages": [("human", query)]})

In [121]:
response

{'messages': [HumanMessage(content='What is the population of India and Pakistan? Add populations of both', id='cf94c10e-3cc6-45cc-8e32-3fd0e089c3c0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_6cYRaR0xTCoQ3ksaQjyThghX', 'function': {'arguments': '{"query": "Population of India"}', 'name': 'search_wikipedia'}, 'type': 'function'}, {'id': 'call_Py25mDaSsxHCTUy4gAROUs0D', 'function': {'arguments': '{"query": "Population of Pakistan"}', 'name': 'search_wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 216, 'total_tokens': 266, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-672fff21-3959-45ba-bce8-c3e606a415f

In [122]:
# Final Response
response['messages'][-1].content

'As of 2024, the population of India is approximately **1.484 billion** (1,484,000,000), and the population of Pakistan is about **241.5 million** (241,499,431). \n\nWhen we add the populations of both countries together, we get a total population of approximately **1.725 billion** (1,725,499,431).'

In [120]:
print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content') and msg.content:
        print(f"Content: {msg.content[:50]}")
    if hasattr(msg, 'name') and msg.name:
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls")
        for tool_call in msg.tool_calls:
            print(tool_call)


Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of India and Pakistan? Add 

--- Message 2 ---
Type: AIMessage
Tool calls
{'name': 'search_wikipedia', 'args': {'query': 'Population of India'}, 'id': 'call_6cYRaR0xTCoQ3ksaQjyThghX', 'type': 'tool_call'}
{'name': 'search_wikipedia', 'args': {'query': 'Population of Pakistan'}, 'id': 'call_Py25mDaSsxHCTUy4gAROUs0D', 'type': 'tool_call'}

--- Message 3 ---
Type: ToolMessage
Content: Page: Demographics of India
Summary: India is the 
Name: search_wikipedia

--- Message 4 ---
Type: ToolMessage
Content: Page: Demographics of Pakistan
Summary: Pakistan h
Name: search_wikipedia

--- Message 5 ---
Type: AIMessage
Tool calls
{'name': 'add_numbers', 'args': {'inputs': '1484000000, 241499431'}, 'id': 'call_W3Mgfc7Mig3rEreYsa2JjzS7', 'type': 'tool_call'}

--- Message 6 ---
Type: ToolMessage
Content: {"result": 1725499431}
Name: add_numbers

--- Message 7 ---
Type: AIMessage
Content: As of 2024, the population

## TEST CASES 

In [124]:
# Test Cases for agent which contains only addition tool
test_cases = [
    {
        "query": "Add 100, 20, and 10.",
        "expected": {"result": 130},
        "description": "Testing Addition tool for numbers string"
    },
    {
        "query": "Add -100, -20, and 10.",
        "expected": {"result": -110},
        "description": "Testing Addition tool for numbers string"
    },
]

In [ ]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = agent_exec.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'search_wikipedia']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        # If we wanna test different variety of tools, we can also use an llm to evaluate.
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing Addition tool for numbers string ---
Query: Add 100, 20, and 10.
Tool Result: 130
Expected Result: 130
✅ Test Passed: Testing Addition tool for numbers string

--- Test Case 2: Testing Addition tool for numbers string ---
Query: Add -100, -20, and 10.
Tool Result: -110
Expected Result: -110
✅ Test Passed: Testing Addition tool for numbers string

Correctly passed tests: ['Testing Addition tool for numbers string', 'Testing Addition tool for numbers string']
